In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.etl_data
""")

print("Volume criado com sucesso!")

In [0]:
import pandas as pd

output_path = "/Volumes/workspace/default/etl_data"

customers = [
    (1, "João Silva", "São Paulo", "SP", "2025-01-10"),
    (2, "Maria Souza", "Recife", "PE", "2025-01-15"),
    (3, "Carlos Lima", "Campina Grande", "PB", "2025-02-01"),
    (4, "Ana Costa", "Curitiba", "PR", "2025-02-10"),
    (5, "Pedro Santos", "João Pessoa", "PB", "2025-03-05")
]

columns = [
    "customer_id",
    "customer_name",
    "city",
    "state",
    "registration_date"
]

df = pd.DataFrame(customers, columns=columns)

df.to_csv(
    f"{output_path}/customers.csv",
    index=False,
    encoding="utf-8"
)

print("Arquivo customers.csv criado com sucesso!")
display(df)

In [0]:
import pandas as pd

output_path = "/Volumes/workspace/default/etl_data"

# FONTE 2: PRODUTOS
products = [
    (101, "Notebook", "Eletrônicos", 3500.00),
    (102, "Mouse", "Periféricos", 80.00),
    (103, "Teclado", "Periféricos", 150.00),
    (104, "Monitor", "Eletrônicos", 1200.00),
    (105, "Cadeira", "Móveis", 750.00),
    (106, "Headset", "Periféricos", 220.00)
]

products_columns = [
    "product_id",
    "product_name",
    "category",
    "unit_price"
]

df_products = pd.DataFrame(products, columns=products_columns)

df_products.to_csv(
    f"{output_path}/products.csv",
    index=False,
    encoding="utf-8"
)

# FONTE 3: VENDAS
sales = [
    (1001, 1, 101, 1, "2025-03-10"),
    (1002, 2, 102, 2, "2025-03-11"),
    (1003, 3, 103, 1, "2025-03-12"),
    (1004, 4, 104, 2, "2025-03-13"),
    (1005, 5, 105, 1, "2025-03-14"),
    (1006, 1, 106, 3, "2025-03-15"),
    (1007, 2, 101, 1, "2025-03-16"),
    (1008, 3, 102, 2, "2025-03-17")
]

sales_columns = [
    "sale_id",
    "customer_id",
    "product_id",
    "quantity",
    "sale_date"
]

df_sales = pd.DataFrame(sales, columns=sales_columns)

df_sales.to_csv(
    f"{output_path}/sales.csv",
    index=False,
    encoding="utf-8"
)

print("Fontes de dados criadas com sucesso!")

print("PRODUTOS")
display(df_products)

print("VENDAS")
display(df_sales)

In [0]:
# Caminho dos arquivos
source_path = "/Volumes/workspace/default/etl_data"

# Leitura dos arquivos utilizando PySpark
df_customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{source_path}/customers.csv")

df_products = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{source_path}/products.csv")

df_sales = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{source_path}/sales.csv")

print("Extração concluída!")

print("CLIENTES")
display(df_customers)

print("PRODUTOS")
display(df_products)

print("VENDAS")
display(df_sales)

In [0]:
from pyspark.sql.functions import col

# Função para análise de qualidade
def quality_check(df, table_name):

    print(f"\n===== QUALITY CHECK: {table_name} =====")

    total_records = df.count()

    print(f"Total de registros: {total_records}")

    print("\nValores nulos por coluna:")

    for column in df.columns:
        null_count = df.filter(
            col(column).isNull()
        ).count()

        print(f"{column}: {null_count}")

    duplicate_count = total_records - df.dropDuplicates().count()

    print(f"\nRegistros duplicados: {duplicate_count}")


# Executando as verificações
quality_check(df_customers, "CUSTOMERS")
quality_check(df_products, "PRODUCTS")
quality_check(df_sales, "SALES")

In [0]:
from pyspark.sql.functions import col, lit, when

# Criando uma cópia de teste com problemas
df_customers_test = (
    df_customers
    .unionByName(df_customers.limit(1))
    .withColumn(
        "city",
        when(col("customer_id") == 2, lit(None))
        .otherwise(col("city"))
    )
)

# Exibindo dados com problemas
display(df_customers_test)

# Executando novamente a validação
quality_check(df_customers_test, "CUSTOMERS_TEST")

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    initcap,
    to_date
)

# Transformação da tabela de clientes
df_customers_silver = (
    df_customers
    # Remover duplicidades pela chave do cliente
    .dropDuplicates(["customer_id"])

    # Remover registros sem informações obrigatórias
    .dropna(subset=[
        "customer_id",
        "customer_name",
        "city",
        "state",
        "registration_date"
    ])

    # Padronização dos campos textuais
    .withColumn("customer_name", initcap(trim(col("customer_name"))))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("state", upper(trim(col("state"))))

    # Conversão da data
    .withColumn(
        "registration_date",
        to_date(col("registration_date"), "yyyy-MM-dd")
    )
)

print("Transformação Silver concluída!")

display(df_customers_silver)

print("Schema da tabela:")
df_customers_silver.printSchema()

print("Total de registros:", df_customers_silver.count())

In [0]:
from pyspark.sql.functions import col, trim, initcap, to_date
from pyspark.sql.types import DecimalType, IntegerType

# =====================================
# TRANSFORMAÇÃO DA TABELA DE PRODUTOS
# =====================================

df_products_silver = (
    df_products
    .dropDuplicates(["product_id"])
    .dropna(subset=[
        "product_id",
        "product_name",
        "category",
        "unit_price"
    ])
    .withColumn("product_name", initcap(trim(col("product_name"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .filter(col("unit_price") > 0)
)

# =====================================
# TRANSFORMAÇÃO DA TABELA DE VENDAS
# =====================================

df_sales_silver = (
    df_sales
    .dropDuplicates(["sale_id"])
    .dropna(subset=[
        "sale_id",
        "customer_id",
        "product_id",
        "quantity",
        "sale_date"
    ])
    .withColumn("sale_id", col("sale_id").cast(IntegerType()))
    .withColumn("customer_id", col("customer_id").cast(IntegerType()))
    .withColumn("product_id", col("product_id").cast(IntegerType()))
    .withColumn("quantity", col("quantity").cast(IntegerType()))
    .withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))
    .filter(col("quantity") > 0)
)

# =====================================
# RESULTADOS
# =====================================

print("PRODUTOS SILVER")
display(df_products_silver)

print("VENDAS SILVER")
display(df_sales_silver)

print("RESUMO")
print("Clientes:", df_customers_silver.count())
print("Produtos:", df_products_silver.count())
print("Vendas:", df_sales_silver.count())

In [0]:
# Criar o schema Silver dentro do catálogo workspace
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

# Gravar tabela de clientes
df_customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.customers")

# Gravar tabela de produtos
df_products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.products")

# Gravar tabela de vendas
df_sales_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.sales")

print("Todas as tabelas Silver foram gravadas!")

In [0]:
%sql
select * from workspace.silver.sales

In [0]:
%sql

select * from workspace.silver.customers

In [0]:
%sql

select * from workspace.silver.products